## 1 - Load anomaly candidates

In [16]:
from pathlib import Path

import numpy as np
import pandas as pd


INPUT_PATH = Path(
    "../data/processed/log_anomaly_candidates.csv"
)

anomaly_candidates_df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["time_window"],
)

anomaly_candidates_df["time_window"] = pd.to_datetime(
    anomaly_candidates_df["time_window"],
    utc=True,
)

print("Loaded:", INPUT_PATH.resolve())
print("Anomaly buckets:", len(anomaly_candidates_df))

display(anomaly_candidates_df.head())

Loaded: E:\AIO\Project\aiops-trainticket-pipeline\data\processed\log_anomaly_candidates.csv
Anomaly buckets: 51


,time_window,analysis_group,service_key,log_count,info_count,warn_count,error_count,unique_template_count,unique_reporter_count,error_ratio,...,warn_count_zero_baseline_spike,prior_error_count,prior_warn_count,has_enough_history,is_volume_spike,is_error_novelty,is_warn_novelty,is_error_burst,is_warn_burst,is_anomaly
0,2022-06-14 13:42:30+00:00,semantic_cluster_21,train-ticket/ts-auth-service,4,3,0,1,4,4,0.250000,...,False,0,0,True,False,True,False,False,False,True
1,2022-06-14 13:42:30+00:00,semantic_cluster_28,train-ticket/ts-auth-service,3,2,0,1,3,2,0.333333,...,False,0,0,True,False,True,False,False,False,True
2,2022-06-14 14:17:00+00:00,semantic_cluster_33,train-ticket/ts-auth-service,1,0,0,1,1,1,1.000000,...,False,0,0,True,False,True,False,False,False,True
3,2022-06-14 14:18:00+00:00,semantic_cluster_12,train-ticket/ts-auth-service,1,0,0,1,1,1,1.000000,...,False,0,0,True,False,True,False,False,False,True
4,2022-06-14 14:18:00+00:00,semantic_cluster_18,train-ticket/ts-auth-service,3,2,0,1,3,2,0.333333,...,False,0,0,True,False,True,False,False,False,True


## 2 - Kiểm tra các cột bắt buộc

In [17]:
EPISODE_GROUP_COLUMNS = [
    "service_key",
    "active_period_id",
    "analysis_group",
]

ANOMALY_FLAG_COLUMNS = [
    "is_volume_spike",
    "is_error_novelty",
    "is_warn_novelty",
    "is_error_burst",
    "is_warn_burst",
]

REQUIRED_COLUMNS = {
    "time_window",
    "service_key",
    "active_period_id",
    "analysis_group",
    "log_count",
    "info_count",
    "warn_count",
    "error_count",
    *ANOMALY_FLAG_COLUMNS,
}

missing_columns = (
    REQUIRED_COLUMNS
    - set(anomaly_candidates_df.columns)
)

assert not missing_columns, (
    f"Missing required columns: {sorted(missing_columns)}"
)

assert anomaly_candidates_df["time_window"].notna().all()
assert anomaly_candidates_df["service_key"].notna().all()
assert anomaly_candidates_df["active_period_id"].notna().all()
assert anomaly_candidates_df["analysis_group"].notna().all()

print("Input validation passed.")

Input validation passed.


## 3 - Chuẩn hóa anomaly flags

In [18]:
def to_boolean(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series
        .astype("string")
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .fillna(False)
        .astype(bool)
    )


for column in ANOMALY_FLAG_COLUMNS:
    anomaly_candidates_df[column] = to_boolean(
        anomaly_candidates_df[column]
    )

display(
    anomaly_candidates_df[
        ANOMALY_FLAG_COLUMNS
    ].sum()
)

is_volume_spike     10
is_error_novelty    28
is_warn_novelty     13
is_error_burst       0
is_warn_burst        0
dtype: int64

## 4 - Sắp xếp và tính khoảng cách giữa các bucket

In [19]:
TIME_BUCKET_SECONDS = 30
EPISODE_GAP_SECONDS = 60

anomaly_buckets_df = (
    anomaly_candidates_df
    .sort_values(
        EPISODE_GROUP_COLUMNS + ["time_window"],
        kind="stable",
    )
    .reset_index(drop=True)
)

anomaly_buckets_df["bucket_end"] = (
    anomaly_buckets_df["time_window"]
    + pd.Timedelta(seconds=TIME_BUCKET_SECONDS)
)

grouped_buckets = anomaly_buckets_df.groupby(
    EPISODE_GROUP_COLUMNS,
    sort=False,
    dropna=False,
)

anomaly_buckets_df["previous_time_window"] = (
    grouped_buckets["time_window"].shift(1)
)

anomaly_buckets_df["gap_seconds"] = (
    anomaly_buckets_df["time_window"]
    - anomaly_buckets_df["previous_time_window"]
).dt.total_seconds()

display(
    anomaly_buckets_df[
        EPISODE_GROUP_COLUMNS
        + [
            "time_window",
            "previous_time_window",
            "gap_seconds",
        ]
    ].head(20)
)

,service_key,active_period_id,analysis_group,time_window,previous_time_window,gap_seconds
0,train-ticket/ts-auth-service,12,semantic_cluster_21,2022-06-14 13:42:30+00:00,NaT,NaN
1,train-ticket/ts-auth-service,12,semantic_cluster_28,2022-06-14 13:42:30+00:00,NaT,NaN
2,train-ticket/ts-auth-service,13,semantic_cluster_12,2022-06-14 14:18:00+00:00,NaT,NaN
3,train-ticket/ts-auth-service,13,semantic_cluster_18,2022-06-14 14:18:00+00:00,NaT,NaN
4,train-ticket/ts-auth-service,13,semantic_cluster_33,2022-06-14 14:17:00+00:00,NaT,NaN
5,train-ticket/ts-auth-service,14,semantic_cluster_24,2022-06-14 18:31:30+00:00,NaT,NaN
6,train-ticket/ts-auth-service,16,semantic_cluster_0,2022-06-14 19:21:00+00:00,NaT,NaN
7,train-ticket/ts-auth-service,16,semantic_cluster_19,2022-06-14 19:21:00+00:00,NaT,NaN
8,train-ticket/ts-auth-service,16,semantic_cluster_27,2022-06-14 19:34:00+00:00,NaT,NaN
9,train-ticket/ts-auth-service,16,semantic_cluster_33,2022-06-14 19:21:00+00:00,NaT,NaN


## 5 - Đánh dấu điểm bắt đầu episode mới

In [20]:
anomaly_buckets_df["is_new_episode"] = (
    anomaly_buckets_df["previous_time_window"].isna()
    | anomaly_buckets_df["gap_seconds"].gt(
        EPISODE_GAP_SECONDS
    )
)

anomaly_buckets_df["episode_number"] = (
    anomaly_buckets_df
    .groupby(
        EPISODE_GROUP_COLUMNS,
        sort=False,
        dropna=False,
    )["is_new_episode"]
    .cumsum()
    .astype("int64")
)

In [21]:
display(
    anomaly_buckets_df[
        EPISODE_GROUP_COLUMNS
        + [
            "time_window",
            "gap_seconds",
            "is_new_episode",
            "episode_number",
        ]
    ].head(30)
)

,service_key,active_period_id,analysis_group,time_window,gap_seconds,is_new_episode,episode_number
0,train-ticket/ts-auth-service,12,semantic_cluster_21,2022-06-14 13:42:30+00:00,NaN,True,1
1,train-ticket/ts-auth-service,12,semantic_cluster_28,2022-06-14 13:42:30+00:00,NaN,True,1
2,train-ticket/ts-auth-service,13,semantic_cluster_12,2022-06-14 14:18:00+00:00,NaN,True,1
3,train-ticket/ts-auth-service,13,semantic_cluster_18,2022-06-14 14:18:00+00:00,NaN,True,1
4,train-ticket/ts-auth-service,13,semantic_cluster_33,2022-06-14 14:17:00+00:00,NaN,True,1
5,train-ticket/ts-auth-service,14,semantic_cluster_24,2022-06-14 18:31:30+00:00,NaN,True,1
6,train-ticket/ts-auth-service,16,semantic_cluster_0,2022-06-14 19:21:00+00:00,NaN,True,1
7,train-ticket/ts-auth-service,16,semantic_cluster_19,2022-06-14 19:21:00+00:00,NaN,True,1
8,train-ticket/ts-auth-service,16,semantic_cluster_27,2022-06-14 19:34:00+00:00,NaN,True,1
9,train-ticket/ts-auth-service,16,semantic_cluster_33,2022-06-14 19:21:00+00:00,NaN,True,1


## 6 - Chuyển anomaly flags thành danh sách tín hiệu

In [22]:
ANOMALY_SIGNAL_NAMES = {
    "is_volume_spike": "volume_spike",
    "is_error_novelty": "error_novelty",
    "is_warn_novelty": "warn_novelty",
    "is_error_burst": "error_burst",
    "is_warn_burst": "warn_burst",
}


def collect_anomaly_signals(frame):
    signals = []

    for column, signal_name in (
        ANOMALY_SIGNAL_NAMES.items()
    ):
        if frame[column].any():
            signals.append(signal_name)

    return signals

## 7 - Tổng hợp các bucket thành episode

In [23]:
episode_group_columns = (
    EPISODE_GROUP_COLUMNS
    + ["episode_number"]
)

episode_rows = []

for episode_keys, episode_buckets in (
    anomaly_buckets_df.groupby(
        episode_group_columns,
        sort=False,
        dropna=False,
    )
):
    (
        service_key,
        active_period_id,
        analysis_group,
        episode_number,
    ) = episode_keys

    start_time = episode_buckets["time_window"].min()
    end_time = episode_buckets["bucket_end"].max()

    log_count = int(
        episode_buckets["log_count"].sum()
    )

    info_count = int(
        episode_buckets["info_count"].sum()
    )

    warn_count = int(
        episode_buckets["warn_count"].sum()
    )

    error_count = int(
        episode_buckets["error_count"].sum()
    )

    finite_robust_z = pd.to_numeric(
        episode_buckets.get(
            "log_count_robust_z",
            pd.Series(dtype=float),
        ),
        errors="coerce",
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    )

    max_log_count_robust_z = (
        float(finite_robust_z.max())
        if finite_robust_z.notna().any()
        else None
    )

    episode_rows.append({
        "service_key": service_key,
        "active_period_id": active_period_id,
        "analysis_group": analysis_group,
        "episode_number": int(episode_number),
        "start_time": start_time,
        "end_time": end_time,
        "duration_seconds": float(
            (end_time - start_time).total_seconds()
        ),
        "anomaly_bucket_count": len(
            episode_buckets
        ),
        "log_count": log_count,
        "info_count": info_count,
        "warn_count": warn_count,
        "error_count": error_count,
        "warn_ratio": (
            warn_count / log_count
            if log_count > 0
            else 0.0
        ),
        "error_ratio": (
            error_count / log_count
            if log_count > 0
            else 0.0
        ),
        "anomaly_signals": collect_anomaly_signals(
            episode_buckets
        ),
        "max_log_count_robust_z": (
            max_log_count_robust_z
        ),
    })


log_episodes_df = (
    pd.DataFrame(episode_rows)
    .sort_values(
        [
            "start_time",
            "service_key",
            "analysis_group",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Input anomaly buckets:",
    len(anomaly_buckets_df),
)
print(
    "Group-level episodes:",
    len(log_episodes_df),
)

display(log_episodes_df.head(30))

Input anomaly buckets: 51
Group-level episodes: 51


,service_key,active_period_id,analysis_group,episode_number,start_time,end_time,duration_seconds,anomaly_bucket_count,log_count,info_count,warn_count,error_count,warn_ratio,error_ratio,anomaly_signals,max_log_count_robust_z
0,train-ticket/ts-auth-service,12,semantic_cluster_21,1,2022-06-14 13:42:30+00:00,2022-06-14 13:43:00+00:00,30.0,1,4,3,0,1,0.000000,0.250000,[error_novelty],NaN
1,train-ticket/ts-auth-service,12,semantic_cluster_28,1,2022-06-14 13:42:30+00:00,2022-06-14 13:43:00+00:00,30.0,1,3,2,0,1,0.000000,0.333333,[error_novelty],NaN
2,train-ticket/ts-auth-service,13,semantic_cluster_33,1,2022-06-14 14:17:00+00:00,2022-06-14 14:17:30+00:00,30.0,1,1,0,0,1,0.000000,1.000000,[error_novelty],NaN
3,train-ticket/ts-auth-service,13,semantic_cluster_12,1,2022-06-14 14:18:00+00:00,2022-06-14 14:18:30+00:00,30.0,1,1,0,0,1,0.000000,1.000000,[error_novelty],NaN
4,train-ticket/ts-auth-service,13,semantic_cluster_18,1,2022-06-14 14:18:00+00:00,2022-06-14 14:18:30+00:00,30.0,1,3,2,0,1,0.000000,0.333333,[error_novelty],NaN
5,train-ticket/ts-auth-service,14,semantic_cluster_24,1,2022-06-14 18:31:30+00:00,2022-06-14 18:32:00+00:00,30.0,1,1,0,1,0,1.000000,0.000000,[warn_novelty],NaN
6,train-ticket/ts-auth-service,16,semantic_cluster_37,1,2022-06-14 19:14:30+00:00,2022-06-14 19:15:00+00:00,30.0,1,2,1,0,1,0.000000,0.500000,[error_novelty],NaN
7,train-ticket/ts-auth-service,16,semantic_cluster_0,1,2022-06-14 19:21:00+00:00,2022-06-14 19:21:30+00:00,30.0,1,2,1,0,1,0.000000,0.500000,[error_novelty],NaN
8,train-ticket/ts-auth-service,16,semantic_cluster_19,1,2022-06-14 19:21:00+00:00,2022-06-14 19:21:30+00:00,30.0,1,1,0,1,0,1.000000,0.000000,[warn_novelty],NaN
9,train-ticket/ts-auth-service,16,semantic_cluster_33,1,2022-06-14 19:21:00+00:00,2022-06-14 19:21:30+00:00,30.0,1,1,0,0,1,0.000000,1.000000,[error_novelty],NaN


## 8 - Tạo episode ID

In [24]:
import hashlib
import json


SCHEMA_VERSION = "1.0.0"
SCENARIO_ID = "ts-auth-mongo_4.4.15_2022-07-13"


def to_rfc3339_utc(value):
    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize("UTC")
    else:
        timestamp = timestamp.tz_convert("UTC")

    return (
        timestamp
        .isoformat()
        .replace("+00:00", "Z")
    )


def stable_episode_id(row):
    identity = {
        "schema_version": SCHEMA_VERSION,
        "signal_source": "logs",
        "scenario_id": SCENARIO_ID,
        "service_key": row["service_key"],
        "active_period_id": str(
            row["active_period_id"]
        ),
        "analysis_group": row["analysis_group"],
        "start": to_rfc3339_utc(
            row["start_time"]
        ),
        "end": to_rfc3339_utc(
            row["end_time"]
        ),
        "anomaly_signals": sorted(
            row["anomaly_signals"]
        ),
    }

    serialized_identity = json.dumps(
        identity,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

    digest = hashlib.sha256(
        serialized_identity.encode("utf-8")
    ).hexdigest()[:16]

    return f"lep_{digest}"


log_episodes_df["episode_id"] = (
    log_episodes_df.apply(
        stable_episode_id,
        axis=1,
    )
)

assert log_episodes_df["episode_id"].is_unique

display(
    log_episodes_df[
        [
            "episode_id",
            "start_time",
            "end_time",
            "service_key",
            "analysis_group",
            "anomaly_bucket_count",
            "anomaly_signals",
        ]
    ].head(30)
)

,episode_id,start_time,end_time,service_key,analysis_group,anomaly_bucket_count,anomaly_signals
0,lep_3018fd86cc77652e,2022-06-14 13:42:30+00:00,2022-06-14 13:43:00+00:00,train-ticket/ts-auth-service,semantic_cluster_21,1,[error_novelty]
1,lep_01062be98cbcd900,2022-06-14 13:42:30+00:00,2022-06-14 13:43:00+00:00,train-ticket/ts-auth-service,semantic_cluster_28,1,[error_novelty]
2,lep_add3a15e0af8facc,2022-06-14 14:17:00+00:00,2022-06-14 14:17:30+00:00,train-ticket/ts-auth-service,semantic_cluster_33,1,[error_novelty]
3,lep_914182274afbbb77,2022-06-14 14:18:00+00:00,2022-06-14 14:18:30+00:00,train-ticket/ts-auth-service,semantic_cluster_12,1,[error_novelty]
4,lep_2e9990dabfed0c35,2022-06-14 14:18:00+00:00,2022-06-14 14:18:30+00:00,train-ticket/ts-auth-service,semantic_cluster_18,1,[error_novelty]
5,lep_37400cec8baa82ff,2022-06-14 18:31:30+00:00,2022-06-14 18:32:00+00:00,train-ticket/ts-auth-service,semantic_cluster_24,1,[warn_novelty]
6,lep_01118b050f838fbc,2022-06-14 19:14:30+00:00,2022-06-14 19:15:00+00:00,train-ticket/ts-auth-service,semantic_cluster_37,1,[error_novelty]
7,lep_0e26a54496bceba8,2022-06-14 19:21:00+00:00,2022-06-14 19:21:30+00:00,train-ticket/ts-auth-service,semantic_cluster_0,1,[error_novelty]
8,lep_6c95c254e56aca20,2022-06-14 19:21:00+00:00,2022-06-14 19:21:30+00:00,train-ticket/ts-auth-service,semantic_cluster_19,1,[warn_novelty]
9,lep_6385456c96ecb213,2022-06-14 19:21:00+00:00,2022-06-14 19:21:30+00:00,train-ticket/ts-auth-service,semantic_cluster_33,1,[error_novelty]


## Cấu hình schema và helper

In [25]:
SCHEMA_NAME = "aiops.log_incident_candidates"
SCHEMA_VERSION = "1.0.0"
PRODUCER_NAME = "03_3_logs_episode_aggregation"
CORRELATOR_VERSION = "1.0.0"

CORRELATION_TOLERANCE_SECONDS = 90

PROJECT_ROOT = (
    Path("..")
    if Path("../data").exists()
    else Path(".")
)


def finite_float_or_none(value):
    if value is None or pd.isna(value):
        return None

    number = float(value)
    return number if np.isfinite(number) else None


def split_service_key(value):
    namespace, name = str(value).split("/", maxsplit=1)

    return {
        "namespace": namespace,
        "name": name,
    }


def candidate_id_from_episode_id(episode_id):
    digest = str(episode_id).removeprefix("lep_")
    return f"lic_{digest}"

## Chuyển log episodes thành candidates

In [26]:
candidate_records = []

for episode in log_episodes_df.itertuples(index=False):
    service = split_service_key(episode.service_key)

    start_time = pd.Timestamp(episode.start_time)
    end_time = pd.Timestamp(episode.end_time)

    candidate_records.append({
        "candidate_id": candidate_id_from_episode_id(
            episode.episode_id
        ),
        "signal_source": "logs",
        "event_time": {
            "start": to_rfc3339_utc(start_time),
            "end": to_rfc3339_utc(end_time),
            "duration_seconds": float(
                (end_time - start_time).total_seconds()
            ),
        },
        "scope": {
            "level": "entity_local",
            "entities": [
                {
                    "role": "application",
                    "service": service,
                }
            ],
        },
        "correlation": {
            "scenario_id": SCENARIO_ID,
            "service_keys": [
                episode.service_key
            ],
            "time_tolerance_seconds": (
                CORRELATION_TOLERANCE_SECONDS
            ),
        },
        "summary": {
            "analysis_group": episode.analysis_group,
            "anomaly_signals": sorted(
                episode.anomaly_signals
            ),
            "anomaly_bucket_count": int(
                episode.anomaly_bucket_count
            ),
            "log_count": int(episode.log_count),
            "info_count": int(episode.info_count),
            "warn_count": int(episode.warn_count),
            "error_count": int(episode.error_count),
            "warn_ratio": float(episode.warn_ratio),
            "error_ratio": float(episode.error_ratio),
            "max_log_count_robust_z": (
                finite_float_or_none(
                    episode.max_log_count_robust_z
                )
            ),
        },
        "evidence": [
            {
                "episode_id": episode.episode_id,
                "active_period_id": str(
                    episode.active_period_id
                ),
                "analysis_group": (
                    episode.analysis_group
                ),
                "event_time": {
                    "start": to_rfc3339_utc(
                        start_time
                    ),
                    "end": to_rfc3339_utc(
                        end_time
                    ),
                },
                "anomaly_signals": sorted(
                    episode.anomaly_signals
                ),
                "counts": {
                    "buckets": int(
                        episode.anomaly_bucket_count
                    ),
                    "logs": int(episode.log_count),
                    "info": int(episode.info_count),
                    "warnings": int(
                        episode.warn_count
                    ),
                    "errors": int(
                        episode.error_count
                    ),
                },
            }
        ],
    })


print("Log incident candidates:", len(candidate_records))

display(
    pd.DataFrame([
        {
            "candidate_id": record["candidate_id"],
            "start": record["event_time"]["start"],
            "end": record["event_time"]["end"],
            "scope": record["scope"]["level"],
            "services": record[
                "correlation"
            ]["service_keys"],
            "signals": record[
                "summary"
            ]["anomaly_signals"],
        }
        for record in candidate_records
    ]).head(30)
)

Log incident candidates: 51


,candidate_id,start,end,scope,services,signals
0,lic_3018fd86cc77652e,2022-06-14T13:42:30Z,2022-06-14T13:43:00Z,entity_local,[train-ticket/ts-auth-service],[error_novelty]
1,lic_01062be98cbcd900,2022-06-14T13:42:30Z,2022-06-14T13:43:00Z,entity_local,[train-ticket/ts-auth-service],[error_novelty]
2,lic_add3a15e0af8facc,2022-06-14T14:17:00Z,2022-06-14T14:17:30Z,entity_local,[train-ticket/ts-auth-service],[error_novelty]
3,lic_914182274afbbb77,2022-06-14T14:18:00Z,2022-06-14T14:18:30Z,entity_local,[train-ticket/ts-auth-service],[error_novelty]
4,lic_2e9990dabfed0c35,2022-06-14T14:18:00Z,2022-06-14T14:18:30Z,entity_local,[train-ticket/ts-auth-service],[error_novelty]
5,lic_37400cec8baa82ff,2022-06-14T18:31:30Z,2022-06-14T18:32:00Z,entity_local,[train-ticket/ts-auth-service],[warn_novelty]
6,lic_01118b050f838fbc,2022-06-14T19:14:30Z,2022-06-14T19:15:00Z,entity_local,[train-ticket/ts-auth-service],[error_novelty]
7,lic_0e26a54496bceba8,2022-06-14T19:21:00Z,2022-06-14T19:21:30Z,entity_local,[train-ticket/ts-auth-service],[error_novelty]
8,lic_6c95c254e56aca20,2022-06-14T19:21:00Z,2022-06-14T19:21:30Z,entity_local,[train-ticket/ts-auth-service],[warn_novelty]
9,lic_6385456c96ecb213,2022-06-14T19:21:00Z,2022-06-14T19:21:30Z,entity_local,[train-ticket/ts-auth-service],[error_novelty]


## Tạo document schema hoàn chỉnh

In [27]:
generated_at = pd.Timestamp.now(tz="UTC")

log_incident_document = {
    "schema_name": SCHEMA_NAME,
    "schema_version": SCHEMA_VERSION,
    "generated_at": to_rfc3339_utc(generated_at),
    "producer": {
        "name": PRODUCER_NAME,
        "pipeline_stage": (
            "log_episode_aggregation"
        ),
        "detector": (
            "robust_baseline_and_novelty"
        ),
        "correlator_version": CORRELATOR_VERSION,
    },
    "time_contract": {
        "normalized_timezone": "UTC",
        "timestamp_format": "RFC3339",
        "interval_semantics": "closed",
        "native_resolution_seconds": int(
            TIME_BUCKET_SECONDS
        ),
        "default_correlation_tolerance_seconds": (
            CORRELATION_TOLERANCE_SECONDS
        ),
    },
    "provenance": {
        "notebook": (
            "notebooks/"
            "03_3_logs_episode_aggregation.ipynb"
        ),
        "input": (
            "data/processed/"
            "log_anomaly_candidates.csv"
        ),
        "scenario_id": SCENARIO_ID,
        "timestamp_normalization": {
            "source_timezone": "Asia/Shanghai",
            "normalized_timezone": "UTC",
            "source_utc_offset_seconds": 28800,
            "applied_offset_seconds": -28800,
            "assumption": True,
        },
    },
    "candidates": candidate_records,
}

print(
    "Schema:",
    log_incident_document["schema_name"],
)
print(
    "Version:",
    log_incident_document["schema_version"],
)
print(
    "Candidates:",
    len(log_incident_document["candidates"]),
)

Schema: aiops.log_incident_candidates
Version: 1.0.0
Candidates: 51


## Validation trước khi lưu

In [28]:
required_candidate_fields = {
    "candidate_id",
    "signal_source",
    "event_time",
    "scope",
    "correlation",
    "summary",
    "evidence",
}

candidate_ids = []

for candidate in log_incident_document["candidates"]:
    missing_fields = (
        required_candidate_fields
        - set(candidate)
    )

    assert not missing_fields, (
        f"Candidate thiếu field: {missing_fields}"
    )

    assert candidate["signal_source"] == "logs"

    candidate_id = candidate["candidate_id"]
    candidate_ids.append(candidate_id)

    assert candidate_id.startswith("lic_")

    start = pd.Timestamp(
        candidate["event_time"]["start"]
    )
    end = pd.Timestamp(
        candidate["event_time"]["end"]
    )

    assert start.tzinfo is not None
    assert end.tzinfo is not None
    assert start <= end

    assert (
        candidate["correlation"]["scenario_id"]
        == SCENARIO_ID
    )

    assert (
        candidate["correlation"]["service_keys"]
    )

    assert candidate["scope"]["entities"]

    for timestamp_field in ["start", "end"]:
        assert candidate[
            "event_time"
        ][timestamp_field].endswith("Z")


assert len(candidate_ids) == len(set(candidate_ids)), (
    "candidate_id bị trùng"
)

# Serialize thử để phát hiện numpy type, NaN hoặc Infinity.
serialized_document = json.dumps(
    log_incident_document,
    ensure_ascii=False,
    indent=2,
    allow_nan=False,
)

assert serialized_document

print("Output validation passed.")

Output validation passed.


## Lưu vào data/outputs

In [29]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOG_INCIDENTS_PATH = (
    OUTPUT_DIR
    / "log_incident_candidates.json"
)

LOG_INCIDENTS_PATH.write_text(
    serialized_document,
    encoding="utf-8",
)

print(
    "Exported:",
    LOG_INCIDENTS_PATH.resolve(),
)
print(
    "Candidates:",
    len(candidate_records),
)
print(
    "File size:",
    LOG_INCIDENTS_PATH.stat().st_size,
    "bytes",
)

Exported: E:\AIO\Project\aiops-trainticket-pipeline\data\outputs\log_incident_candidates.json
Candidates: 51
File size: 87142 bytes
